In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "enviroment_bj").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

## Embedding for DeepSeek4

In [ ]:
from src.transformer_modules.embedding_module import *

# ============================================================
# Smoke Test
# ============================================================

config = EmbeddingConfig(
    vocab_size=16000,
    d_model=256,
    pad_token_id=0,
    max_seq_len=512,
    embedding_dropout=0.0,
    scale_embeddings=False,
    init_std=0.02,
    tie_word_embeddings=True,)

embedding = TokenEmbedding(config)

B, T = 4, 128

input_ids = torch.randint(
    low=0,
    high=config.vocab_size,
    size=(B, T),
    dtype=torch.long,)

hidden_states = embedding(input_ids)

print("input_ids shape:", input_ids.shape)
print("hidden_states shape:", hidden_states.shape)
print("hidden_states dtype:", hidden_states.dtype)

assert hidden_states.shape == (B, T, config.d_model)
assert torch.isfinite(hidden_states).all()

if config.pad_token_id is not None:
    pad_row = embedding.weight[config.pad_token_id]
    assert torch.allclose(pad_row, torch.zeros_like(pad_row))

print("TokenEmbedding module OK.")

input_ids shape: torch.Size([4, 128])
hidden_states shape: torch.Size([4, 128, 256])
hidden_states dtype: torch.float32
TokenEmbedding module OK.


---

# Causal MHA 

In [ ]:
from src.transformer_modules.mha_baseline import *

config = CausalMHAConfig(
    d_model=256,
    n_heads=4,
    head_dim=None,
    attention_dropout=0.0,
    residual_dropout=0.0,
    use_bias=False,
    use_rope=True,
    rope_theta=10000.0,
    rotary_dim=64,
    max_seq_len=512,
    init_std=0.02,)

attn = CausalMultiHeadAttention(config)

x = torch.randn(2, 128, 256)

out, weights = attn(
    x,
    attention_mask=None,
    position_ids=None,
    start_pos=0,
    need_weights=True)

print(out.shape)      # [2, 128, 256]
print(weights.shape)  # [2, 4, 128, 128]

torch.Size([2, 128, 256])
torch.Size([2, 4, 128, 128])


---

# SwiGLU / MLP

In [ ]:
from src.transformer_modules.SwiGLU import * 

# ============================================================
# Smoke test
# ============================================================

mlp_config = SwiGLUMLPConfig(
    d_model=256,
    hidden_dim=None,
    expansion_factor=4.0,
    multiple_of=1,
    dropout=0.0,
    use_bias=False,
    init_std=0.02,)

mlp = SwiGLUMLP(mlp_config)

x = torch.randn(2, 128, 256)

out = mlp(x)

print("x shape:", x.shape)
print("out shape:", out.shape)
print("hidden_dim:", mlp.hidden_dim)

assert out.shape == x.shape
assert torch.isfinite(out).all()

print("SwiGLUMLP OK.")

x shape: torch.Size([2, 128, 256])
out shape: torch.Size([2, 128, 256])
hidden_dim: 1024
SwiGLUMLP OK.


---

# Baseline Transformer Block 

In [ ]:
from src.transformer_modules.transformer_block import * 

# ============================================================
# Embedding -> TransformerBlock
# ============================================================

block_config = TransformerBlockConfig(
    d_model=256,
    rms_norm_eps=1e-6,

    n_heads=4,
    head_dim=64,
    attention_dropout=0.0,
    residual_dropout=0.0,
    use_attention_bias=False,
    use_rope=True,
    rope_theta=10000.0,
    rotary_dim=64,
    max_seq_len=128,

    mlp_hidden_dim=None,
    mlp_expansion_factor=4.0,
    mlp_multiple_of=1,
    mlp_dropout=0.0,
    use_mlp_bias=False,

    init_std=0.02,
)


B, T = 2, 128
vocab_size = 16000
d_model = 256

embedding_config = EmbeddingConfig(
    vocab_size=vocab_size,
    d_model=d_model,
    pad_token_id=0,
    max_seq_len=T,
    embedding_dropout=0.0,
    scale_embeddings=False,
    init_std=0.02,
    tie_word_embeddings=True,
)

embedding = TokenEmbedding(embedding_config)
block = TransformerBlock(block_config)

input_ids = torch.randint(1, vocab_size, (B, T), dtype=torch.long)
attention_mask = torch.ones(B, T, dtype=torch.long)

hidden_states = embedding(input_ids)
out = block(hidden_states, attention_mask=attention_mask)

print("hidden_states:", hidden_states.shape)
print("out:", out.shape)

assert hidden_states.shape == (B, T, d_model)
assert out.shape == (B, T, d_model)
assert torch.isfinite(out).all()

print("Embedding -> TransformerBlock pipeline OK.")

hidden_states: torch.Size([2, 128, 256])
out: torch.Size([2, 128, 256])
Embedding -> TransformerBlock pipeline OK.


---

# Mini Large Model for Baseline

In [ ]:
from src.transformer_modules.transformer import * 

# ============================================================
# Smoke test: random input_ids
# ============================================================

config = MiniCausalLMConfig(
    vocab_size=16000,
    d_model=256,
    n_layers=2,

    pad_token_id=0,
    max_seq_len=128,

    embedding_dropout=0.0,
    scale_embeddings=False,
    tie_word_embeddings=True,

    rms_norm_eps=1e-6,

    n_heads=4,
    head_dim=64,
    attention_dropout=0.0,
    residual_dropout=0.0,
    use_attention_bias=False,
    use_rope=True,
    rope_theta=10000.0,
    rotary_dim=64,

    mlp_hidden_dim=None,
    mlp_expansion_factor=4.0,
    mlp_multiple_of=1,
    mlp_dropout=0.0,
    use_mlp_bias=False,

    init_std=0.02,
)

model = MiniCausalLM(config)

B, T = 2, 128

input_ids = torch.randint(
    low=1,
    high=config.vocab_size,
    size=(B, T),
    dtype=torch.long)

labels = torch.randint(
    low=1,
    high=config.vocab_size,
    size=(B, T),
    dtype=torch.long)

outputs = model(
    input_ids=input_ids,
    labels=labels,
    need_weights=True)

logits = outputs["logits"]
loss = outputs["loss"]
aux = outputs["aux"]

print("logits:", logits.shape)
print("loss:", loss)
print("num attn layers:", len(aux["attn_weights"]))
print("attn layer 0:", aux["attn_weights"][0].shape)

assert logits.shape == (B, T, config.vocab_size)
assert loss is not None
assert loss.dim() == 0
assert torch.isfinite(loss)
assert len(aux["attn_weights"]) == config.n_layers
assert aux["attn_weights"][0].shape == (B, config.n_heads, T, T)
assert model.lm_head.weight is model.embedding.weight

print("MiniCausalLM baseline OK.")

logits: torch.Size([2, 128, 16000])
loss: tensor(9.7476, grad_fn=<NllLossBackward0>)
num attn layers: 2
attn layer 0: torch.Size([2, 4, 128, 128])
MiniCausalLM baseline OK.


---

# DeepSeekv4 Modules


# HCA: Heavily Compressed Attention

In [ ]:
from src.deepseek_hca_attention import * 

B, T, Dh = 2, 130, 64
m = 16

compressor = HCATokenCompressor(
    compression_factor=m,
    head_dim=Dh,
    init_std=0.02)

C = torch.randn(B, T, Dh)
Z = torch.randn(B, T, Dh)
attention_mask = torch.ones(B, T, dtype=torch.long)

compressed_C, compressed_valid_mask, compressed_position_ids = compressor(
    C=C,
    Z=Z,
    attention_mask=attention_mask,
    start_pos=0)

S = math.ceil(T / m)

print("compressed_C:", compressed_C.shape)
print("compressed_valid_mask:", compressed_valid_mask.shape)
print("compressed_position_ids:", compressed_position_ids)

assert compressed_C.shape == (B, S, Dh)
assert compressed_valid_mask.shape == (B, S)
assert compressed_position_ids.shape == (S,)

print("HCATokenCompressor OK.")

compressed_C: torch.Size([2, 9, 64])
compressed_valid_mask: torch.Size([2, 9])
compressed_position_ids: tensor([ 15,  31,  47,  63,  79,  95, 111, 127, 129])
HCATokenCompressor OK.


---

# CSA: Compressed Sparse Attention

In [ ]:
from src.deepseek_csa_attention import * 

# ============================================================
# Canonical CSA smoke test
# ============================================================

csa_config = CSAConfig(
    d_model=256,
    n_heads=4,
    head_dim=64,

    compression_factor=8,
    top_k=4,
    window_size=32,

    indexer_dim=32,
    n_indexer_heads=2,
    query_compression_dim=64,

    attention_dropout=0.0,
    residual_dropout=0.0,
    use_bias=False,

    use_rope=True,
    rope_theta=10000.0,
    rotary_dim=64,

    max_seq_len=512,
    init_std=0.02,
)

csa = CSAAttention(csa_config)

B, T, D = 2, 128, 256
x = torch.randn(B, T, D)
attention_mask = torch.ones(B, T, dtype=torch.long)

out, aux = csa(
    x,
    attention_mask=attention_mask,
    position_ids=None,
    start_pos=0,
    need_weights=True,
)

S = math.ceil(T / csa_config.compression_factor)
K = min(csa_config.top_k, S)

print("out:", out.shape)
print("global_attn_weights:", aux["global_attn_weights"].shape)
print("local_attn_weights:", aux["local_attn_weights"].shape)
print("topk_indices:", aux["topk_indices"].shape)
print("topk_mask:", aux["topk_mask"].shape)
print("compressed_valid_mask:", aux["compressed_valid_mask"].shape)
print("index_scores:", aux["index_scores"].shape)

assert out.shape == (B, T, D)
assert aux["global_attn_weights"].shape == (B, csa_config.n_heads, T, K)
assert aux["local_attn_weights"].shape == (B, csa_config.n_heads, T, T)
assert aux["topk_indices"].shape == (B, T, K)
assert aux["topk_mask"].shape == (B, T, K)
assert aux["compressed_valid_mask"].shape == (B, S)
assert aux["index_scores"].shape == (B, T, S)

assert torch.isfinite(out).all()
assert torch.isfinite(aux["global_attn_weights"]).all()
assert torch.isfinite(aux["local_attn_weights"]).all()
assert torch.isfinite(aux["index_scores"]).all()

print("Canonical CSAAttention smoke test OK.")

out: torch.Size([2, 128, 256])
global_attn_weights: torch.Size([2, 4, 128, 4])
local_attn_weights: torch.Size([2, 4, 128, 128])
topk_indices: torch.Size([2, 128, 4])
topk_mask: torch.Size([2, 128, 4])
compressed_valid_mask: torch.Size([2, 16])
index_scores: torch.Size([2, 128, 16])
Canonical CSAAttention smoke test OK.


In [49]:
# ============================================================
# Top-k causal check
# ============================================================

m = csa_config.compression_factor
topk_indices = aux["topk_indices"]
topk_mask = aux["topk_mask"]

for t in range(T):
    query_block = t // m

    selected = topk_indices[:, t, :]
    valid = topk_mask[:, t, :]

    if valid.any():
        assert (selected[valid] < query_block).all()

print("Canonical CSA top-k causal check OK.")

Canonical CSA top-k causal check OK.


---

# mHC / residuals

In [ ]:
from src.mHC_residuals import * 

# ============================================================
# mHC smoke tests
# ============================================================

B, T, D = 2, 8, 64
n_hc = 4

x = torch.randn(B, T, D)

X = expand_residual_stream(x, n_hc=n_hc, mode="first")

config = ManifoldHyperConnectionConfig(
    d_model=D,
    n_hc=n_hc,
    sinkhorn_iters=20,
    eps=1e-6,
    init_alpha=1e-3,
    dynamic=True,
    init_std=0.02,
)

mhc = ManifoldHyperConnection(config)

def dummy_sublayer(x_sub):
    return 0.1 * x_sub

X_next, aux = mhc(
    X,
    sublayer=dummy_sublayer,
    return_aux=True,)

print("X:", X.shape)
print("X_next:", X_next.shape)
print("A:", aux["A"].shape)
print("B:", aux["B"].shape)
print("C:", aux["C"].shape)
print("x_sub:", aux["x_sub"].shape)
print("y_sub:", aux["y_sub"].shape)

assert X.shape == (B, T, n_hc, D)
assert X_next.shape == (B, T, n_hc, D)
assert aux["A"].shape == (B, T, 1, n_hc)
assert aux["B"].shape == (B, T, n_hc, n_hc)
assert aux["C"].shape == (B, T, n_hc, 1)
assert aux["x_sub"].shape == (B, T, D)
assert aux["y_sub"].shape == (B, T, D)

assert torch.isfinite(X_next).all()
assert torch.isfinite(aux["A"]).all()
assert torch.isfinite(aux["B"]).all()
assert torch.isfinite(aux["C"]).all()

print("mHC smoke test OK.")

X: torch.Size([2, 8, 4, 64])
X_next: torch.Size([2, 8, 4, 64])
A: torch.Size([2, 8, 1, 4])
B: torch.Size([2, 8, 4, 4])
C: torch.Size([2, 8, 4, 1])
x_sub: torch.Size([2, 8, 64])
y_sub: torch.Size([2, 8, 64])
mHC smoke test OK.


In [55]:
# ============================================================
# Approx residual-normal initialization check
# ============================================================

config_static = ManifoldHyperConnectionConfig(
    d_model=D,
    n_hc=n_hc,
    dynamic=False,
    sinkhorn_iters=50,
    init_alpha=0.0,)

mhc_static = ManifoldHyperConnection(config_static)

X = expand_residual_stream(x, n_hc=n_hc, mode="first")

def F(x_sub):
    return 0.1 * x_sub

X_next, aux = mhc_static(X, sublayer=F, return_aux=True)

# Since A selects stream 0 and C injects mainly into stream 0,
# stream 0 should be close to x + F(x), up to small leakage from sigmoid/Sinkhorn.
expected_stream0 = x + F(x)

print("mean abs stream0 error:", (X_next[:, :, 0, :] - expected_stream0).abs().mean().item())
print("mean abs inactive streams:", X_next[:, :, 1:, :].abs().mean().item())

assert torch.isfinite(X_next).all()

print("mHC static residual-like initialization check OK.")

mean abs stream0 error: 0.0022263354621827602
mean abs inactive streams: 0.0006514514680020511
mHC static residual-like initialization check OK.


In [56]:
x = torch.randn(B, T, D, requires_grad=True)
X = expand_residual_stream(x, n_hc=n_hc, mode="first")

mhc = ManifoldHyperConnection(config)

X_next = mhc(
    X,
    sublayer=lambda x_sub: x_sub ** 2,
    return_aux=False,
)

loss = X_next.sum()
loss.backward()

assert x.grad is not None
assert torch.isfinite(x.grad).all()

for name, param in mhc.named_parameters():
    assert param.grad is not None, f"Missing grad for {name}"
    assert torch.isfinite(param.grad).all(), f"Non-finite grad for {name}"

print("mHC backward OK.")

mHC backward OK.


---

# DeepSeekMoE

In [ ]:
from src.deepseek_moe import * 

# ============================================================
# DeepSeekMoE smoke test
# ============================================================

moe_config = DeepSeekMoEConfig(
    d_model=64,

    num_experts=8,
    top_k=2,

    expert_hidden_dim=256,
    expert_expansion_factor=4.0,
    expert_multiple_of=1,

    shared_experts=1,
    shared_hidden_dim=None,
    shared_expansion_factor=4.0,

    router_score_fn="sqrt_softplus",
    normalize_topk_weights=True,

    router_jitter_noise=0.0,

    dropout=0.0,
    use_bias=False,
    init_std=0.02,

    balance_loss_weight=0.01,
)

moe = DeepSeekMoE(moe_config)

B, T, D = 2, 16, 64
x = torch.randn(B, T, D)

out, aux = moe(x, return_aux=True)

print("out:", out.shape)
print("router_logits:", aux["router_logits"].shape)
print("router_scores:", aux["router_scores"].shape)
print("topk_indices:", aux["topk_indices"].shape)
print("topk_weights:", aux["topk_weights"].shape)
print("expert_counts:", aux["expert_counts"].shape)
print("expert_fraction:", aux["expert_fraction"].shape)
print("router_entropy:", aux["router_entropy"])
print("balance_loss:", aux["balance_loss"])

assert out.shape == x.shape
assert aux["router_logits"].shape == (B, T, moe_config.num_experts)
assert aux["router_scores"].shape == (B, T, moe_config.num_experts)
assert aux["topk_indices"].shape == (B, T, moe_config.top_k)
assert aux["topk_weights"].shape == (B, T, moe_config.top_k)
assert aux["expert_counts"].shape == (moe_config.num_experts,)
assert aux["expert_fraction"].shape == (moe_config.num_experts,)

assert torch.isfinite(out).all()
assert torch.isfinite(aux["router_logits"]).all()
assert torch.isfinite(aux["router_scores"]).all()
assert torch.isfinite(aux["topk_weights"]).all()
assert torch.isfinite(aux["router_entropy"])
assert torch.isfinite(aux["balance_loss"])

assert (aux["topk_weights"] >= 0).all()

if moe_config.normalize_topk_weights:
    assert torch.allclose(
        aux["topk_weights"].sum(dim=-1),
        torch.ones_like(aux["topk_weights"].sum(dim=-1)),
        atol=1e-6,
        rtol=1e-6,
    )

print("DeepSeekMoE smoke test OK.")

out: torch.Size([2, 16, 64])
router_logits: torch.Size([2, 16, 8])
router_scores: torch.Size([2, 16, 8])
topk_indices: torch.Size([2, 16, 2])
topk_weights: torch.Size([2, 16, 2])
expert_counts: torch.Size([8])
expert_fraction: torch.Size([8])
router_entropy: tensor(2.0782, grad_fn=<NegBackward0>)
balance_loss: tensor(5.4932e-06)
DeepSeekMoE smoke test OK.


In [69]:
# ============================================================
# Drop-in FFN replacement check
# ============================================================

B, T, D = 2, 16, 64

norm = RMSNorm(dim=D)
moe = DeepSeekMoE(
    DeepSeekMoEConfig(
        d_model=D,
        num_experts=4,
        top_k=2,
        expert_hidden_dim=128,
        shared_experts=1,
        router_score_fn="sqrt_softplus",
        normalize_topk_weights=True,
        dropout=0.0,
        use_bias=False,
        init_std=0.02,
        balance_loss_weight=0.01,
    )
)

x = torch.randn(B, T, D)

residual = x
ffn_out, aux = moe(norm(x), return_aux=True)
x_next = residual + ffn_out

assert x_next.shape == x.shape
assert torch.isfinite(x_next).all()
assert torch.isfinite(aux["balance_loss"])

print("MoE drop-in FFN replacement OK.")

MoE drop-in FFN replacement OK.


---


# MTP

In [ ]:
from src.deepseek_mtp import * 

# ============================================================
# MTP smoke test
# ============================================================

B, T, D, V = 2, 16, 64, 128
mtp_depth = 3

mtp_config = MTPConfig(
    d_model=D,
    vocab_size=V,
    mtp_depth=mtp_depth,

    hidden_dim=None,
    use_mtp_transform=True,
    activation="silu",

    dropout=0.0,
    use_bias=False,
    init_std=0.02,

    tie_with_lm_head=False,
    mtp_loss_weight=0.3,

    pad_token_id=0,
)

mtp_head = MultiTokenPredictionHead(mtp_config)

hidden_states = torch.randn(B, T, D)

mtp_labels = torch.randint(
    low=1,
    high=V,
    size=(B, mtp_depth, T),
    dtype=torch.long,
)

# Simulate ignored/padded positions
mtp_labels[:, :, -2:] = 0

outputs = mtp_head(
    hidden_states=hidden_states,
    mtp_labels=mtp_labels,
    return_aux=True,
)

print("mtp_logits:", outputs["mtp_logits"].shape)
print("mtp_loss:", outputs["mtp_loss"])
print("raw_mtp_loss:", outputs["aux"]["raw_mtp_loss"])
print("mtp_loss_per_depth:", outputs["aux"]["mtp_loss_per_depth"].shape)

assert outputs["mtp_logits"].shape == (B, mtp_depth, T, V)
assert outputs["mtp_loss"] is not None
assert outputs["mtp_loss"].dim() == 0
assert torch.isfinite(outputs["mtp_loss"])
assert outputs["aux"]["mtp_loss_per_depth"].shape == (mtp_depth,)
assert torch.isfinite(outputs["aux"]["mtp_loss_per_depth"]).all()

print("MTP smoke test OK.")

mtp_logits: torch.Size([2, 3, 16, 128])
mtp_loss: tensor(1.4556, grad_fn=<MulBackward0>)
raw_mtp_loss: tensor(4.8520, grad_fn=<SumBackward0>)
mtp_loss_per_depth: torch.Size([3])
MTP smoke test OK.


In [75]:
# ============================================================
# MTP tying smoke test
# ============================================================

lm_head = nn.Linear(D, V, bias=False)

mtp_config_tied = MTPConfig(
    d_model=D,
    vocab_size=V,
    mtp_depth=2,
    use_mtp_transform=False,
    tie_with_lm_head=True,
    mtp_loss_weight=0.3,
    pad_token_id=0,
)

mtp_head_tied = MultiTokenPredictionHead(mtp_config_tied)

mtp_head_tied.tie_weights(lm_head.weight)

for head in mtp_head_tied.heads:
    assert head.weight is lm_head.weight

hidden_states = torch.randn(B, T, D)

outputs = mtp_head_tied(hidden_states)

assert outputs["mtp_logits"].shape == (B, 2, T, V)

print("MTP weight tying OK.")

MTP weight tying OK.


In [76]:
# ============================================================
# MTP backward check
# ============================================================

mtp_head = MultiTokenPredictionHead(mtp_config)

hidden_states = torch.randn(B, T, D, requires_grad=True)

mtp_labels = torch.randint(
    low=1,
    high=V,
    size=(B, mtp_depth, T),
    dtype=torch.long,
)
mtp_labels[:, :, -2:] = 0

outputs = mtp_head(
    hidden_states=hidden_states,
    mtp_labels=mtp_labels,
    return_aux=True,
)

loss = outputs["mtp_loss"]
loss.backward()

assert hidden_states.grad is not None
assert torch.isfinite(hidden_states.grad).all()

for name, param in mtp_head.named_parameters():
    assert param.grad is not None, f"Missing grad for {name}"
    assert torch.isfinite(param.grad).all(), f"Non-finite grad for {name}"

print("MTP backward OK.")

MTP backward OK.


---

## Full DeepSeekv4

In [ ]:
from src.mini_deepseek_class import * 

# ============================================================
# Smoke test 1: Dense + MHA, no mHC, no MTP
# ============================================================

cfg = DeepSeekV4LMConfig(
    vocab_size=216,
    d_model=64,
    n_layers=2,
    max_seq_len=64,
    pad_token_id=0,
    ignore_index=-100,

    attention_type="mha",
    n_heads=4,
    head_dim=16,
    rotary_dim=16,

    ffn_type="dense",
    mlp_hidden_dim=128,

    use_mhc=False,
    use_mtp=False,

    tie_word_embeddings=True,
    init_std=0.02,
)

model = DeepSeekV4LM(cfg)

B, T = 2, 32
input_ids = torch.randint(1, cfg.vocab_size, (B, T), dtype=torch.long)
labels = torch.randint(1, cfg.vocab_size, (B, T), dtype=torch.long)

out = model(
    input_ids=input_ids,
    labels=labels,
    return_aux=True,
    need_weights=True,
)

print("logits:", out["logits"].shape)
print("loss:", out["loss"])

assert out["logits"].shape == (B, T, cfg.vocab_size)
assert out["loss"] is not None
assert torch.isfinite(out["loss"])
assert len(out["aux"]["blocks"]) == cfg.n_layers

out["loss"].backward()

for name, param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        assert torch.isfinite(param.grad).all(), f"Non-finite grad in {name}"

print("DeepSeekV4LM dense+mha smoke OK.")

logits: torch.Size([2, 32, 216])
loss: tensor(5.4033, grad_fn=<NllLossBackward0>)
DeepSeekV4LM dense+mha smoke OK.


In [191]:
# ============================================================
# Smoke test 2: CSA + MoE + MTP, no mHC
# ============================================================

cfg = DeepSeekV4LMConfig(
    vocab_size=216,
    d_model=64,
    n_layers=2,
    max_seq_len=64,
    pad_token_id=0,
    ignore_index=-100,

    attention_type="csa",
    n_heads=4,
    head_dim=16,
    rotary_dim=16,
    compression_factor=4,
    top_k_blocks=2,
    window_size=8,
    indexer_dim=8,
    n_indexer_heads=2,
    query_compression_dim=16,
    use_grouped_output_projection=True,
    output_projection_groups=2,
    use_attention_sink=True,
    use_indexer_score_bias=False,
    use_separate_local_kv=True,

    ffn_type="moe",
    num_experts=4,
    top_k_experts=2,
    expert_hidden_dim=128,
    shared_experts=1,
    shared_hidden_dim=128,
    router_type="learned",
    router_score_fn="sqrt_softplus",
    balance_loss_weight=0.01,
    sequence_balance_loss_weight=0.01,

    use_mhc=True,

    use_mtp=True,
    mtp_depth=2,
    mtp_hidden_dim=64,
    mtp_loss_weight=0.3,
    mtp_tie_with_lm_head=False,

    tie_word_embeddings=True,
    init_std=0.02,
)

model = DeepSeekV4LM(cfg)

B, T = 2, 32
input_ids = torch.randint(1, cfg.vocab_size, (B, T), dtype=torch.long)
labels = input_ids.clone()
labels[:, :-1] = input_ids[:, 1:]
labels[:, -1] = cfg.ignore_index

outputs = model(
    input_ids=input_ids,
    labels=labels,
    mtp_labels=None,       # auto build from input_ids
    return_aux=True,
    need_weights=True,
)

print("logits:", outputs["logits"].shape)
print("loss:", outputs["loss"])
print("lm_loss:", outputs["lm_loss"])
print("mtp_loss:", outputs["mtp_loss"])
print("moe_aux_loss:", outputs["moe_aux_loss"])

assert outputs["logits"].shape == (B, T, cfg.vocab_size)
assert outputs["loss"] is not None
assert outputs["lm_loss"] is not None
assert outputs["mtp_loss"] is not None
assert outputs["moe_aux_loss"] is not None
assert torch.isfinite(outputs["loss"])
assert len(outputs["aux"]["blocks"]) == cfg.n_layers
assert "mtp" in outputs["aux"]

outputs["loss"].backward()

assert any(
    p.grad is not None
    for p in model.parameters()
), "No gradients were produced."

print("DeepSeekV4LM CSA+MoE+MTP smoke OK.")

logits: torch.Size([2, 32, 216])
loss: tensor(6.9783, grad_fn=<AddBackward0>)
lm_loss: tensor(5.3656, grad_fn=<NllLossBackward0>)
mtp_loss: tensor(1.6126, grad_fn=<MulBackward0>)
moe_aux_loss: tensor(0.0001)
DeepSeekV4LM CSA+MoE+MTP smoke OK.


In [ ]:
from data.syntethic_long_context_retrieval import *

CFG = SyntheticRetrievalConfig(
    # Dataset size
    num_train_examples=50_000,
    num_val_examples=5_000,

    # Sequence length
    block_size=64,              # debe ser <= model.config.max_seq_len
    min_filler_tokens=8,
    max_filler_tokens=32,

    # Task structure
    num_keys_per_example=4,
    vocab_filler_size=68,      
    num_key_types=64,
    num_value_types=64,

    # DataLoader
    batch_size=32,
    num_workers=0,

    # Reproducibility
    seed=42)

train_loader, val_loader, tokenizer = create_synthetic_retrieval_dataloaders(
        cfg=CFG,
        use_mtp=False,)


config = DeepSeekV4LMConfig(
    vocab_size=tokenizer.vocab_size,
    d_model=256,
    n_layers=4,
    max_seq_len=1024,
    pad_token_id=tokenizer.pad_id,

    attention_type="hybrid",
    attention_pattern=("csa", "hca"),

    n_heads=4,
    head_dim=64,
    rotary_dim=64,

    compression_factor=8,       # CSA compression
    hca_compression_factor=16,  # HCA heavier compression
    top_k_blocks=8,
    window_size=32,

    ffn_type="moe",
    use_mhc=True,
    use_mtp=True,

    balance_loss_weight=0.01,
    sequence_balance_loss_weight=0.01
)

model = DeepSeekV4LM(config)

B, T = 2, 32

input_ids = torch.randint(
    1,
    config.vocab_size,
    (B, T),
    dtype=torch.long,
)

labels = input_ids.clone()
labels[:, :-1] = input_ids[:, 1:]
labels[:, -1] = config.ignore_index

outputs = model(
    input_ids=input_ids,
    labels=labels,
    mtp_labels=None,
    return_aux=True,
    need_weights=True,
)

print("logits:", outputs["logits"].shape)
print("loss:", outputs["loss"])
print("lm_loss:", outputs["lm_loss"])
print("mtp_loss:", outputs["mtp_loss"])
print("moe_aux_loss:", outputs["moe_aux_loss"])

assert outputs["logits"].shape == (B, T, config.vocab_size)
assert outputs["loss"] is not None
assert outputs["lm_loss"] is not None
assert outputs["mtp_loss"] is not None

# Correcto: como balance_loss_weight=0 y sequence_balance_loss_weight=0,
# no debe entrar como componente de la loss total.
assert outputs["moe_aux_loss"] is None

assert torch.isfinite(outputs["loss"])
assert len(outputs["aux"]["blocks"]) == config.n_layers
assert "mtp" in outputs["aux"]

outputs["loss"].backward()

assert any(
    p.grad is not None
    for p in model.parameters()
), "No gradients were produced."

print("DeepSeekV4LM hybrid CSA/HCA + MoE + MTP smoke OK.")

logits: torch.Size([2, 32, 216])
loss: tensor(7.0553, grad_fn=<AddBackward0>)
lm_loss: tensor(5.4426, grad_fn=<NllLossBackward0>)
mtp_loss: tensor(1.6127, grad_fn=<MulBackward0>)
moe_aux_loss: None
DeepSeekV4LM hybrid CSA/HCA + MoE + MTP smoke OK.
